# Data Cleaning & Preparation Notebook

## Research Question
**Is biodiversity loss in beef-exporting countries associated with the volume of beef exports to the United Kingdom?**

## Data Sources
| Dataset | File | Description                                                                                                             |
|---------|------|-------------------------------------------------------------------------------------------------------------------------|
| BIO | `bio-raw.csv` | Biodiversity Intactness Index (BII) per country, scenario, and year. Historical: 1970–2014; SSP projections: 2015–2050. |
| FAOSTAT | `FAOSTAT_data_en_3-31-2026.csv` | Annual bilateral trade: beef & veal export quantities to the UK, by reporting country, 2001–2024.                       |

## Variables
| Variable | Role | Description |
|----------|------|-------------|
| `bii` | Predictor (raw) | Biodiversity Intactness Index (0–1; 1 = fully intact) |
| `bii_loss` | Predictor (derived) | 1 − BII (higher = more biodiversity loss) |
| `uk_import_qty_t` | Outcome (raw) | Beef export quantity to UK (tonnes) |
| `log_uk_import_qty_t` | Outcome (transformed) | log(1 + uk_import_qty_t) — corrects right-skew |

## Timescale Rationale
- Historical BIO data runs to **2014**; FAOSTAT starts at **2001**.
- Overlap for fully observed data: **2001–2014** (14 years).
- To extend to 2024 we stitch in **SSP2** ("middle of the road") BIO projections for 2015–2024.

## Outputs
All saved to `data/clean/`:
| File | Description |
|------|-------------|
| `bio_clean.csv` | BII per country × year × scenario, ISO3 extracted |
| `fao_clean.csv` | FAOSTAT tidy: iso3, country, year, uk_import_qty_t |
| `panel_2001_2014.csv` | Panel (country × year), 2001–2014, historical BIO only |
| `crosssection_avg.csv` | Cross-sectional country averages, 2001–2014 |
| `panel_extended_2001_2024.csv` | Extended panel: historical + SSP2 BIO, 2001–2024 |

---
## 0 · Setup

In [52]:
import os
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 120)

# ── Paths ──────────────────────────────────────────────────────────────────
# This notebook lives in data/, so paths are relative to that folder.
DATA = os.path.dirname(os.path.abspath("__file__"))  # data/
OUT  = os.path.join(DATA, "clean")
os.makedirs(OUT, exist_ok=True)

BIO_PATH = os.path.join(DATA, "bio-raw.csv")
FAO_PATH = os.path.join(DATA, "FAOSTAT_data_en_3-31-2026.csv")

# ── Parameters ─────────────────────────────────────────────────────────────
# Minimum total exports to UK (tonnes) across all years to keep a country.
MIN_TOTAL_TRADE_T = 10

# Importing country we focus on.
IMPORT_COUNTRY = "United Kingdom of Great Britain and Northern Ireland"

# SSP scenario used to extend BIO beyond 2014.
EXTENSION_SCENARIO = "ssp2rcp4p5messageglobiom"  # "middle of the road"


### ISO-3 Country-Code Helper

We use a comprehensive built-in lookup table that covers every FAOSTAT
reporter-country name (including FAO-specific spellings like "Trkiye" and
"China, mainland").  `pycountry` is used only as a fallback for any names
not already in the table.

> **Why not rely on `pycountry` alone?**  Two reasons:
> 1. `pycountry` may not be installed in the Jupyter kernel (the old code
>    caught `ImportError` silently via `except Exception: pass`, causing
>    *all* non-override lookups to return `None`).
> 2. Some FAO names ("China, mainland", "Netherlands (Kingdom of the)",
>    "Trkiye") have no match in the ISO standard even with fuzzy search.

In [53]:
# ── Complete FAOSTAT-name → ISO3 lookup ────────────────────────────────────
# Covers every Reporter Countries value in the FAOSTAT beef-trade extract.
FAOSTAT_ISO3 = {
    "Argentina":                    "ARG",
    "Australia":                    "AUS",
    "Austria":                      "AUT",
    "Bahrain":                      "BHR",
    "Belgium":                      "BEL",
    "Botswana":                     "BWA",
    "Brazil":                       "BRA",
    "Brunei Darussalam":            "BRN",
    "Bulgaria":                     "BGR",
    "China, mainland":              "CHN",
    "China, Taiwan Province of":    "TWN",
    "Colombia":                     "COL",
    "Croatia":                      "HRV",
    "Cyprus":                       "CYP",
    "Czechia":                      "CZE",
    "Denmark":                      "DNK",
    "Dominican Republic":           "DOM",
    "Estonia":                      "EST",
    "Finland":                      "FIN",
    "France":                       "FRA",
    "Germany":                      "DEU",
    "Ghana":                        "GHA",
    "Greece":                       "GRC",
    "Hungary":                      "HUN",
    "Iceland":                      "ISL",
    "India":                        "IND",
    "Indonesia":                    "IDN",
    "Ireland":                      "IRL",
    "Israel":                       "ISR",
    "Italy":                        "ITA",
    "Jamaica":                      "JAM",
    "Kyrgyzstan":                   "KGZ",
    "Latvia":                       "LVA",
    "Lebanon":                      "LBN",
    "Lithuania":                    "LTU",
    "Luxembourg":                   "LUX",
    "Malta":                        "MLT",
    "Mozambique":                   "MOZ",
    "Namibia":                      "NAM",
    "Netherlands (Kingdom of the)": "NLD",
    "New Zealand":                  "NZL",
    "Norway":                       "NOR",
    "Philippines":                  "PHL",
    "Poland":                       "POL",
    "Portugal":                     "PRT",
    "Republic of Korea":            "KOR",
    "Romania":                      "ROU",
    "Russian Federation":           "RUS",
    "Singapore":                    "SGP",
    "Slovakia":                     "SVK",
    "Slovenia":                     "SVN",
    "South Africa":                 "ZAF",
    "Spain":                        "ESP",
    "Sweden":                       "SWE",
    "Switzerland":                  "CHE",
    "Trinidad and Tobago":          "TTO",
    "Trkiye":                       "TUR",
    "Türkiye":                      "TUR",
    "Ukraine":                      "UKR",
    "United Arab Emirates":         "ARE",
    "United States of America":     "USA",
    "Uruguay":                      "URY",
    "Viet Nam":                     "VNM",
    "Zimbabwe":                     "ZWE",
}

# Track whether pycountry is available (checked once, not per-row).
try:
    import pycountry as _pycountry
    _HAS_PYCOUNTRY = True
except ImportError:
    _HAS_PYCOUNTRY = False
    print("NOTE: pycountry is not installed — using built-in FAOSTAT_ISO3 table only.")


def name_to_iso3(name: str):
    """Return ISO 3166-1 alpha-3 for a country name string.

    Resolution order:
      1. Built-in FAOSTAT_ISO3 dictionary  (fast, no dependencies)
      2. pycountry exact match             (if installed)
      3. pycountry fuzzy search            (if installed)
    """
    # 1) Built-in table — covers all known FAOSTAT names
    if name in FAOSTAT_ISO3:
        return FAOSTAT_ISO3[name]

    # 2) pycountry fallback for any new / unknown names
    if _HAS_PYCOUNTRY:
        try:
            result = _pycountry.countries.get(name=name)
            if result:
                return result.alpha_3
            hits = _pycountry.countries.search_fuzzy(name)
            if hits:
                return hits[0].alpha_3
        except LookupError:
            pass

    return None

---
# Part A — Biodiversity (BIO) Data
## 1 · Load & Explore Raw BIO Data

The BIO dataset contains multiple indicators (BII, land-use shares, etc.).
Since our research question focuses on **biodiversity loss**, we retain only
the **Biodiversity Intactness Index (`bii`)** and derive `bii_loss = 1 − bii`.
All other indicators (pastureland, crops, highintensityag, etc.) are dropped*.

In [54]:
bio_raw = pd.read_csv(BIO_PATH, low_memory=False)

print(f"Shape: {bio_raw.shape}")
print(f"Columns: {list(bio_raw.columns)}")
print(f"Year range: {bio_raw['year'].min()} – {bio_raw['year'].max()}  (dtype: {bio_raw['year'].dtype})")
print(f"Scenarios: {bio_raw['scenario'].unique()}")
print(f"Variables (indicators): {bio_raw['variable'].unique()}")
bio_raw.head()

Shape: (376992, 8)
Columns: ['_id', 'area_code', 'lower_uncertainty', 'scenario', 'upper_uncertainty', 'value', 'variable', 'year']
Year range: 1970 – 2050  (dtype: int64)
Scenarios: <StringArray>
[          'ssp4rcp6p0gcam',   'ssp5rcp8p5remindmagpie',               'historical',          'ssp1rcp2p6image',
 'ssp2rcp4p5messageglobiom',            'ssp3rcp7p0aim']
Length: 6, dtype: str
Variables (indicators): <StringArray>
['bii', 'crops', 'highintensityag', 'hpd', 'pastureland', 'qualitynatural', 'urbanextent']
Length: 7, dtype: str


,_id,area_code,lower_uncertainty,scenario,upper_uncertainty,value,variable,year
0,344002,001-002-202-018-BWA,NaN,ssp4rcp6p0gcam,NaN,0.759763,bii,2044
1,344003,001-002-202-018-LSO,NaN,ssp4rcp6p0gcam,NaN,0.553173,bii,2044
2,344006,001-002-202-018-ZAF,NaN,ssp4rcp6p0gcam,NaN,0.623727,bii,2044
3,344014,001-009-054-PNG,NaN,ssp4rcp6p0gcam,NaN,0.986023,bii,2044
4,344023,001-009-057-PLW,NaN,ssp4rcp6p0gcam,NaN,NaN,bii,2044


### 1.1 · Validate: null counts & data quality

In [55]:
print("Null counts per column:")
print(bio_raw.isna().sum())
print(f"\nRows with null area_code: {bio_raw['area_code'].isna().sum():,}")
print(f"Rows with null value:     {bio_raw['value'].isna().sum():,}")
print(f"Total rows:               {len(bio_raw):,}")

Null counts per column:
_id                       0
area_code              1386
lower_uncertainty    371217
scenario                  0
upper_uncertainty    371217
value                 38412
variable                  0
year                      0
dtype: int64

Rows with null area_code: 1,386
Rows with null value:     38,412
Total rows:               376,992


### 1.2 · Understand the `area_code` structure
Each `area_code` is a dash-separated hierarchy (continent-region-…-ISO3).
The **last segment** is the ISO 3166-1 alpha-3 country code — that's what we need.

In [56]:
print("Sample area_codes:")
print(bio_raw["area_code"].dropna().unique()[:10])

Sample area_codes:
<StringArray>
['001-002-202-018-BWA', '001-002-202-018-LSO', '001-002-202-018-ZAF',     '001-009-054-PNG',     '001-009-057-PLW',
     '001-009-061-COK',     '001-009-061-NIU',     '001-009-061-PYF', '001-019-419-005-COL', '001-019-419-005-ECU']
Length: 10, dtype: str


## 2 · Clean BIO Data

Steps:
1. **Drop rows** with null `area_code` (no country identity → useless).
2. **Extract ISO3** from the last segment of `area_code`.
3. **Keep only valid 3-letter codes** (drop supra-national aggregates like numeric codes).
4. **Filter to `bii` indicator only** — the sole predictor for our research question.
5. **Drop unneeded columns**: `_id` (database artefact), `lower_uncertainty` / `upper_uncertainty` (all NA for BII).
6. **Drop rows with null `value`** (no measurement → can't model).
7. **Pivot wide** so `bii` becomes its own column.
8. **Derive `bii_loss`** = 1 − BII (higher = more degraded, easier to interpret as a predictor).
9. **Drop countries with no BII data** — small islands/territories that lack BII and don't export beef to the UK.

In [57]:
# Step 2.1 – Drop rows with no area code
bio = bio_raw.dropna(subset=["area_code"]).copy()
print(f"After dropping null area_code: {bio.shape[0]:,} rows  (removed {len(bio_raw) - len(bio):,})")

# Step 2.2 – Extract ISO3 from the last segment of area_code
bio["iso3"] = bio["area_code"].str.split("-").str[-1].str.upper()

# Step 2.3 – Keep only standard 3-letter ISO3 codes
bio = bio[bio["iso3"].str.match(r"^[A-Z]{3}$", na=False)]
print(f"After filtering to valid ISO3 codes: {bio.shape[0]:,} rows  |  {bio['iso3'].nunique()} countries")

# Step 2.4 – Keep only the BII indicator (our sole biodiversity predictor)
bio = bio[bio["variable"] == "bii"].copy()
print(f"After filtering to BII indicator: {bio.shape[0]:,} rows")

# Step 2.5 – Drop columns not needed for modelling
bio.drop(columns=["_id", "area_code", "lower_uncertainty", "upper_uncertainty", "variable"], inplace=True)

# Step 2.6 – Drop rows with missing BII value
before = len(bio)
bio.dropna(subset=["value"], inplace=True)
print(f"After dropping null values: {bio.shape[0]:,} rows  (removed {before - len(bio):,})")

# Step 2.7 – Rename value column
bio.rename(columns={"value": "bii"}, inplace=True)

# Step 2.8 – Derive bii_loss = 1 - bii  (higher = more degraded)
bio["bii_loss"] = 1 - bio["bii"]

INDICATOR_COLS = ["bii", "bii_loss"]

print(f"\nCleaned BIO: {bio.shape}")
print(f"  Scenarios:  {bio['scenario'].nunique()} — {sorted(bio['scenario'].unique())}")
print(f"  Countries:  {bio['iso3'].nunique()}")
print(f"  Years:      {bio['year'].min()} – {bio['year'].max()}")
bio.head()

After dropping null area_code: 375,606 rows  (removed 1,386)
After filtering to valid ISO3 codes: 332,640 rows  |  240 countries
After filtering to BII indicator: 47,520 rows
After dropping null values: 39,798 rows  (removed 7,722)

Cleaned BIO: (39798, 5)
  Scenarios:  6 — ['historical', 'ssp1rcp2p6image', 'ssp2rcp4p5messageglobiom', 'ssp3rcp7p0aim', 'ssp4rcp6p0gcam', 'ssp5rcp8p5remindmagpie']
  Countries:  201
  Years:      1970 – 2050


,scenario,bii,year,iso3,bii_loss
0,ssp4rcp6p0gcam,0.759763,2044,BWA,0.240237
1,ssp4rcp6p0gcam,0.553173,2044,LSO,0.446827
2,ssp4rcp6p0gcam,0.623727,2044,ZAF,0.376273
3,ssp4rcp6p0gcam,0.986023,2044,PNG,0.013977
8,ssp4rcp6p0gcam,0.716595,2044,COL,0.283405


In [58]:
# Data is already in wide format (one row per iso3 × year × scenario)
# since we filtered to the single BII indicator above.

bio_wide = bio.copy()

print(f"BIO shape: {bio_wide.shape}")
print(f"Columns: {list(bio_wide.columns)}")
print(f"Countries: {bio_wide['iso3'].nunique()}")
bio_wide.head(10)

BIO shape: (39798, 5)
Columns: ['scenario', 'bii', 'year', 'iso3', 'bii_loss']
Countries: 201


,scenario,bii,year,iso3,bii_loss
0,ssp4rcp6p0gcam,0.759763,2044,BWA,0.240237
1,ssp4rcp6p0gcam,0.553173,2044,LSO,0.446827
2,ssp4rcp6p0gcam,0.623727,2044,ZAF,0.376273
3,ssp4rcp6p0gcam,0.986023,2044,PNG,0.013977
8,ssp4rcp6p0gcam,0.716595,2044,COL,0.283405
9,ssp4rcp6p0gcam,0.789611,2044,ECU,0.210389
10,ssp4rcp6p0gcam,0.893540,2044,PER,0.106460
11,ssp4rcp6p0gcam,0.313319,2044,URY,0.686681
12,ssp4rcp6p0gcam,0.856686,2044,BLZ,0.143314
13,ssp4rcp6p0gcam,0.616668,2044,NIC,0.383332


### 2.1 · Validate: BIO cleaning results

In [59]:
print("Null counts per column after cleaning:")
print(bio_wide[INDICATOR_COLS].isna().sum())

print(f"\nHistorical rows: {(bio_wide['scenario'] == 'historical').sum():,}")
print(f"SSP2 rows:       {(bio_wide['scenario'] == EXTENSION_SCENARIO).sum():,}")

print("\nDescriptive stats — historical scenario:")
bio_wide[bio_wide["scenario"] == "historical"][INDICATOR_COLS].describe().round(4)

Null counts per column after cleaning:
bii         0
bii_loss    0
dtype: int64

Historical rows: 3,618
SSP2 rows:       7,236

Descriptive stats — historical scenario:


,bii,bii_loss
count,3618.0000,3618.0000
mean,0.7622,0.2378
std,0.1597,0.1597
min,0.2982,0.0000
25%,0.6377,0.0958
50%,0.7623,0.2377
75%,0.9042,0.3623
max,1.0000,0.7018


In [60]:
# Save cleaned BIO
bio_wide.to_csv(os.path.join(OUT, "bio_clean.csv"), index=False)
print(f"Saved → data/clean/bio_clean.csv  ({bio_wide.shape[0]:,} rows, {bio_wide.shape[1]} cols)")

Saved → data/clean/bio_clean.csv  (39,798 rows, 5 cols)


### 2.2 · Historical BIO slice (for merging later)
We isolate the **historical** scenario — these are observed data (not model projections).
This is the most reliable subset for analysis.

In [61]:
bio_hist = (
    bio_wide[bio_wide["scenario"] == "historical"]
    .drop(columns=["scenario"])
    .copy()
)
print(f"Historical BIO: {bio_hist.shape[0]:,} rows")
print(f"Year range: {bio_hist['year'].min()} – {bio_hist['year'].max()}")
print(f"Countries:  {bio_hist['iso3'].nunique()}")
bio_hist.head()

Historical BIO: 3,618 rows
Year range: 1970 – 2014
Countries:  201


,bii,year,iso3,bii_loss
283989,0.966643,1970,DZA,0.033357
283990,1.000000,1970,EGY,0.000000
283991,0.943249,1970,BFA,0.056751
283992,0.912323,1970,MLI,0.087677
283993,0.846239,1970,MRT,0.153761


---
# Part B — FAOSTAT Trade Data
## 3 · Load & Explore Raw FAOSTAT Data

> **Note:** After cleaning, only ~32 countries remain with non-negligible
> UK beef exports. Trade volumes are heavily concentrated in a few countries
> (Ireland, Brazil, Netherlands) — log-transformation is used to reduce
> skewness and the influence of dominant exporters.

In [62]:
fao_raw = pd.read_csv(FAO_PATH, encoding="utf-8-sig")

print(f"Shape: {fao_raw.shape}")
print(f"Columns: {list(fao_raw.columns)}")
print(f"Year range: {fao_raw['Year'].min()} – {fao_raw['Year'].max()}")
print(f"Partner countries: {fao_raw['Partner Countries'].unique()}")
print(f"Items: {fao_raw['Item'].unique()}")
print(f"Elements: {fao_raw['Element'].unique()}")
print(f"Reporter countries (exporters): {fao_raw['Reporter Countries'].nunique()}")
fao_raw.head()

Shape: (763, 16)
Columns: ['Domain Code', 'Domain', 'Reporter Country Code (M49)', 'Reporter Countries', 'Partner Country Code (M49)', 'Partner Countries', 'Element Code', 'Element', 'Item Code (CPC)', 'Item', 'Year Code', 'Year', 'Unit', 'Value', 'Flag', 'Flag Description']
Year range: 2001 – 2024
Partner countries: <StringArray>
['United Kingdom of Great Britain and Northern Ireland']
Length: 1, dtype: str
Items: <StringArray>
['Beef and veal preparations nes']
Length: 1, dtype: str
Elements: <StringArray>
['Export quantity']
Length: 1, dtype: str
Reporter countries (exporters): 67


,Domain Code,Domain,Reporter Country Code (M49),Reporter Countries,Partner Country Code (M49),Partner Countries,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description
0,TM,Detailed trade matrix,32,Argentina,826,United Kingdom of Great Britain and Northern I...,5910,Export quantity,F0875,Beef and veal preparations nes,2001,2001,t,5652.0,A,Official figure
1,TM,Detailed trade matrix,32,Argentina,826,United Kingdom of Great Britain and Northern I...,5910,Export quantity,F0875,Beef and veal preparations nes,2002,2002,t,6247.0,A,Official figure
2,TM,Detailed trade matrix,32,Argentina,826,United Kingdom of Great Britain and Northern I...,5910,Export quantity,F0875,Beef and veal preparations nes,2003,2003,t,6065.0,A,Official figure
3,TM,Detailed trade matrix,32,Argentina,826,United Kingdom of Great Britain and Northern I...,5910,Export quantity,F0875,Beef and veal preparations nes,2004,2004,t,9796.0,A,Official figure
4,TM,Detailed trade matrix,32,Argentina,826,United Kingdom of Great Britain and Northern I...,5910,Export quantity,F0875,Beef and veal preparations nes,2005,2005,t,6830.0,A,Official figure


### 3.1 · Validate: FAOSTAT data quality

In [63]:
print("Null counts:")
print(fao_raw.isna().sum())
print(f"\nFlag descriptions (data quality):")
print(fao_raw["Flag Description"].value_counts())
print(f"\nAll rows import to: {fao_raw['Partner Countries'].unique()}")

Null counts:
Domain Code                    0
Domain                         0
Reporter Country Code (M49)    0
Reporter Countries             0
Partner Country Code (M49)     0
Partner Countries              0
Element Code                   0
Element                        0
Item Code (CPC)                0
Item                           0
Year Code                      0
Year                           0
Unit                           0
Value                          0
Flag                           0
Flag Description               0
dtype: int64

Flag descriptions (data quality):
Flag Description
Official figure                        761
Value imputed by a receiving agency      2
Name: count, dtype: int64

All rows import to: <StringArray>
['United Kingdom of Great Britain and Northern Ireland']
Length: 1, dtype: str


## 4 · Clean FAOSTAT Data

Steps:
1. **Filter** for the target importing country (United Kingdom).
2. **Select & rename** columns for clarity.
3. **Drop zero-trade rows** (no useful information).
4. **Map country names → ISO3** using built-in lookup + `pycountry` fallback.
5. **Remove negligible trade partners** (< 10 tonnes total across all years).
6. **Log-transform** the outcome variable (trade volumes are heavily right-skewed).

In [64]:
# Step 4.1 – Filter for UK imports
fao = fao_raw[fao_raw["Partner Countries"] == IMPORT_COUNTRY].copy()
print(f"After filtering to UK imports: {fao.shape[0]} rows")

# Step 4.2 – Select and rename columns
fao = fao[[
    "Reporter Countries", "Partner Countries",
    "Item", "Element", "Year", "Unit", "Value", "Flag Description",
]].rename(columns={
    "Reporter Countries": "country",
    "Partner Countries":  "import_country",
    "Item":               "commodity",
    "Element":            "measure",
    "Year":               "year",
    "Unit":               "unit",
    "Value":              "uk_import_qty_t",
    "Flag Description":   "data_quality",
})

# Step 4.3 – Drop zero-trade rows
before = len(fao)
fao = fao[fao["uk_import_qty_t"] > 0].copy()
print(f"After dropping zero-trade rows: {fao.shape[0]} rows  (removed {before - len(fao)})")

# Step 4.4 – Map country names to ISO3
fao["iso3"] = fao["country"].map(name_to_iso3)

unmapped = fao[fao["iso3"].isna()]["country"].unique()
if len(unmapped):
    print(f"WARNING: Could not map to ISO3: {unmapped}")
else:
    print(f"All countries mapped to ISO3 ✓  ({fao['iso3'].nunique()} unique countries, {len(fao)} rows)")

# Show a sample from different countries (head() only shows Argentina because it's first alphabetically)
fao.groupby("country").head(1).head(10)

After filtering to UK imports: 763 rows
After dropping zero-trade rows: 675 rows  (removed 88)
All countries mapped to ISO3 ✓  (60 unique countries, 675 rows)


,country,import_country,commodity,measure,year,unit,uk_import_qty_t,data_quality,iso3
0,Argentina,United Kingdom of Great Britain and Northern I...,Beef and veal preparations nes,Export quantity,2001,t,5652.00,Official figure,ARG
16,Australia,United Kingdom of Great Britain and Northern I...,Beef and veal preparations nes,Export quantity,2010,t,7.00,Official figure,AUS
19,Austria,United Kingdom of Great Britain and Northern I...,Beef and veal preparations nes,Export quantity,2003,t,77.00,Official figure,AUT
41,Bahamas,United Kingdom of Great Britain and Northern I...,Beef and veal preparations nes,Export quantity,2021,t,9.07,Official figure,BHS
43,Bahrain,United Kingdom of Great Britain and Northern I...,Beef and veal preparations nes,Export quantity,2014,t,0.17,Official figure,BHR
49,Belgium,United Kingdom of Great Britain and Northern I...,Beef and veal preparations nes,Export quantity,2001,t,63.00,Official figure,BEL
73,Botswana,United Kingdom of Great Britain and Northern I...,Beef and veal preparations nes,Export quantity,2001,t,184.00,Official figure,BWA
80,Brazil,United Kingdom of Great Britain and Northern I...,Beef and veal preparations nes,Export quantity,2001,t,51535.00,Official figure,BRA
105,Bulgaria,United Kingdom of Great Britain and Northern I...,Beef and veal preparations nes,Export quantity,2001,t,1.00,Official figure,BGR
130,Canada,United Kingdom of Great Britain and Northern I...,Beef and veal preparations nes,Export quantity,2001,t,20.00,Official figure,CAN


In [65]:
# Step 4.5 – Remove negligible trade partners
total_by_country = fao.groupby("country")["uk_import_qty_t"].sum()
keep_countries   = total_by_country[total_by_country >= MIN_TOTAL_TRADE_T].index
removed_countries = set(fao["country"].unique()) - set(keep_countries)

fao = fao[fao["country"].isin(keep_countries)].copy()
print(f"Removed {len(removed_countries)} countries with < {MIN_TOTAL_TRADE_T}t total trade: {removed_countries}")
print(f"Remaining: {fao['country'].nunique()} exporting countries")

# Sort and reset index
fao.sort_values(["iso3", "year"], inplace=True)
fao.reset_index(drop=True, inplace=True)

# Step 4.6 – Log-transform outcome (handles right-skew)
fao["log_uk_import_qty_t"] = np.log1p(fao["uk_import_qty_t"])

print(f"\nCleaned FAOSTAT: {fao.shape}")
print(f"Year range: {fao['year'].min()} – {fao['year'].max()}")
print(f"Unique exporters: {fao['iso3'].nunique()}")

Removed 23 countries with < 10t total trade: {'Malta', 'Singapore', 'Bahrain', 'Israel', 'Kyrgyzstan', 'Ghana', 'Viet Nam', 'Switzerland', 'China, mainland', 'Jamaica', 'Philippines', 'Türkiye', 'India', 'Bahamas', 'Mozambique', 'Trinidad and Tobago', 'Republic of Korea', 'Finland', 'Lebanon', 'Russian Federation', 'Dominican Republic', 'Cyprus', 'Luxembourg'}
Remaining: 37 exporting countries

Cleaned FAOSTAT: (609, 10)
Year range: 2001 – 2024
Unique exporters: 37


### 4.1 · Validate: top exporters to the UK

In [66]:
top_exporters = (
    fao.groupby(["iso3", "country"])["uk_import_qty_t"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
)
print("Top 15 beef exporters to the UK (total tonnes, 2001-2024):")
print(top_exporters.to_string())

Top 15 beef exporters to the UK (total tonnes, 2010-2020):
iso3  country                     
BRA   Brazil                          867900.18
IRL   Ireland                         863080.69
ARG   Argentina                        68284.10
DNK   Denmark                          57467.02
FRA   France                           50332.97
URY   Uruguay                          36874.08
NLD   Netherlands (Kingdom of the)     32935.81
DEU   Germany                          27875.95
BEL   Belgium                          23682.11
POL   Poland                           22961.83
SWE   Sweden                           18375.90
ROU   Romania                           6590.78
ITA   Italy                             4218.70
ESP   Spain                             2380.22
USA   United States of America          1784.55


In [67]:
# Save cleaned FAOSTAT
fao.to_csv(os.path.join(OUT, "fao_clean.csv"), index=False)
print(f"Saved -> data/clean/fao_clean.csv  ({fao.shape[0]:,} rows)")

Saved -> data/clean/fao_clean.csv  (609 rows)


---
# Part C — Merge & Build Analysis-Ready Datasets

We join BIO indicators with FAOSTAT trade data on `(iso3, year)` to create
panel and cross-sectional datasets ready for regression.

## 5 · Panel-Building Helper

In [68]:
def build_panel(bio_slice, fao_df, label):
    # Aggregate FAOSTAT by (iso3, year, country) in case of multiple commodity rows
    fao_agg = (
        fao_df.groupby(["iso3", "year", "country"], as_index=False)
              .agg(uk_import_qty_t=("uk_import_qty_t", "sum"))
    )
    fao_agg["log_uk_import_qty_t"] = np.log1p(fao_agg["uk_import_qty_t"])

    # Inner join: only country-years present in BOTH datasets
    merged = bio_slice.merge(fao_agg, on=["iso3", "year"], how="inner")
    merged.sort_values(["iso3", "year"], inplace=True)
    merged.reset_index(drop=True, inplace=True)

    # Drop rows with NaN in any model column
    model_cols = INDICATOR_COLS + ["uk_import_qty_t"]
    before = len(merged)
    merged.dropna(subset=model_cols, inplace=True)
    dropped = before - len(merged)

    # Reorder columns: identifiers -> outcome -> predictors
    id_cols = ["iso3", "country", "year"]
    y_cols  = ["uk_import_qty_t", "log_uk_import_qty_t"]
    merged  = merged[id_cols + y_cols + INDICATOR_COLS]

    n_countries = merged["iso3"].nunique()
    n_years     = merged["year"].nunique()
    print(f"\n{label}")
    print(f"  Shape: {merged.shape}  |  {n_countries} countries x {n_years} years")
    if dropped:
        print(f"  Dropped {dropped} rows with NaN in model columns")
    return merged

## 6 · Panel 2001–2014 (Historical BIO Only)
This is the **primary analysis dataset** — both BIO and FAOSTAT are fully observed (no projections).

In [69]:
panel_hist = build_panel(
    bio_hist[bio_hist["year"].between(2001, 2014)],
    fao[fao["year"].between(2001, 2014)],
    "Panel 2001-2014 (historical BIO)",
)

panel_hist.to_csv(os.path.join(OUT, "panel_2001_2014.csv"), index=False)
print(f"Saved -> data/clean/panel_2001_2014.csv")
panel_hist.head(10)


Panel 2001-2014 (historical BIO)
  Shape: (340, 7)  |  37 countries x 14 years
Saved -> data/clean/panel_2001_2014.csv


,iso3,country,year,uk_import_qty_t,log_uk_import_qty_t,bii,bii_loss
0,ARE,United Arab Emirates,2005,16.0,2.833213,1.000000,0.000000
1,ARG,Argentina,2001,5652.0,8.639942,0.761125,0.238875
2,ARG,Argentina,2002,6247.0,8.740017,0.759830,0.240170
3,ARG,Argentina,2003,6065.0,8.710455,0.757173,0.242827
4,ARG,Argentina,2004,9796.0,9.189831,0.752349,0.247651
5,ARG,Argentina,2005,6830.0,8.829226,0.750718,0.249282
6,ARG,Argentina,2006,5070.0,8.531293,0.743890,0.256110
7,ARG,Argentina,2007,6994.0,8.852951,0.738162,0.261838
8,ARG,Argentina,2008,7447.0,8.915701,0.737284,0.262716
9,ARG,Argentina,2009,9301.0,9.137985,0.737940,0.262060


### 6.1 · Validate: panel coverage

In [70]:
print(f"Countries in panel: {sorted(panel_hist['iso3'].unique())}")
print(f"Years in panel:     {sorted(panel_hist['year'].unique())}")
print(f"\nObservations per year:")
print(panel_hist.groupby("year")["iso3"].count())

Countries in panel: ['ARE', 'ARG', 'AUS', 'AUT', 'BEL', 'BGR', 'BRA', 'BWA', 'CAN', 'CZE', 'DEU', 'DNK', 'ESP', 'EST', 'FRA', 'GRC', 'HRV', 'HUN', 'IRL', 'ITA', 'LTU', 'LVA', 'NAM', 'NLD', 'NOR', 'NZL', 'POL', 'PRT', 'ROU', 'SVK', 'SVN', 'SWE', 'UKR', 'URY', 'USA', 'ZAF', 'ZWE']
Years in panel:     [np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014)]

Observations per year:
year
2001    20
2002    21
2003    18
2004    23
2005    25
2006    24
2007    26
2008    25
2009    22
2010    25
2011    25
2012    29
2013    27
2014    30
Name: iso3, dtype: int64


## 7 · Cross-Sectional Dataset (Country Averages 2001–2014)
One row per country — averages of all variables over 2001–2014.


In [71]:
cs = (
    panel_hist
    .groupby(["iso3", "country"])
    .agg(
        years_observed=("year", "count"),
        **{col: (col, "mean") for col in INDICATOR_COLS},
        uk_import_qty_t=("uk_import_qty_t", "mean"),
        log_uk_import_qty_t=("log_uk_import_qty_t", "mean"),
    )
    .reset_index()
    .sort_values("uk_import_qty_t", ascending=False)
    .reset_index(drop=True)
)

print(f"Cross-sectional shape: {cs.shape}  ({cs.shape[0]} countries)")
cs.to_csv(os.path.join(OUT, "crosssection_avg.csv"), index=False)
print(f"Saved -> data/clean/crosssection_avg.csv")

# Display sorted by UK import volume
display_cols = ["iso3", "country", "uk_import_qty_t", "bii", "bii_loss"]
cs[[c for c in display_cols if c in cs.columns]]

Cross-sectional shape: (37, 7)  (37 countries)
Saved -> data/clean/crosssection_avg.csv


,iso3,country,uk_import_qty_t,bii,bii_loss
0,BRA,Brazil,46461.510714,0.776145,0.223855
1,IRL,Ireland,42609.078571,0.435662,0.564338
2,ARG,Argentina,4874.100000,0.738276,0.261724
3,URY,Uruguay,2633.253571,0.345444,0.654556
4,DNK,Denmark,2450.992857,0.451385,0.548615
5,NLD,Netherlands (Kingdom of the),1841.621429,0.583324,0.416676
6,FRA,France,1377.464286,0.612949,0.387051
7,BEL,Belgium,905.957143,0.631431,0.368569
8,DEU,Germany,883.335714,0.673900,0.326100
9,SWE,Sweden,642.964286,0.939300,0.060700


## 8 · Extended Panel 2001–2024 (Historical + SSP2 Projections)

To extend coverage beyond 2014 we stitch in **SSP2** ("middle of the road") BIO projections for 2015–2024.

**2015–2024 BIO values are modelled, not observed**

In [72]:
# SSP2 BIO slice for 2015-2024
bio_ssp2 = (
    bio_wide[
        (bio_wide["scenario"] == EXTENSION_SCENARIO) &
        (bio_wide["year"].between(2015, 2024))
    ]
    .drop(columns=["scenario"])
    .copy()
)
print(f"SSP2 BIO slice: {bio_ssp2.shape[0]:,} rows  |  years {bio_ssp2['year'].min()}-{bio_ssp2['year'].max()}")

# Combine: historical 2001-2014 + SSP2 2015-2024
bio_combined = pd.concat([
    bio_hist[bio_hist["year"].between(2001, 2014)],
    bio_ssp2,
], ignore_index=True)

panel_ext = build_panel(bio_combined, fao, "Extended panel 2001-2024 (historical + SSP2)")

# Attach bio_source flag
bio_src = bio_combined[["iso3", "year"]].copy()
bio_src["bio_source"] = np.where(bio_combined["year"] <= 2014, "historical", "ssp2_projected")
panel_ext = panel_ext.merge(bio_src.drop_duplicates(), on=["iso3", "year"], how="left")

panel_ext.to_csv(os.path.join(OUT, "panel_extended_2001_2024.csv"), index=False)
print(f"Saved -> data/clean/panel_extended_2001_2024.csv")
panel_ext.head(10)

SSP2 BIO slice: 2,010 rows  |  years 2015-2024

Extended panel 2001-2024 (historical + SSP2)
  Shape: (609, 7)  |  37 countries x 24 years
Saved -> data/clean/panel_extended_2010_2020.csv


,iso3,country,year,uk_import_qty_t,log_uk_import_qty_t,bii,bii_loss,bio_source
0,ARE,United Arab Emirates,2005,16.0,2.833213,1.000000,0.000000,historical
1,ARE,United Arab Emirates,2017,8.5,2.251292,1.000000,0.000000,ssp2_projected
2,ARG,Argentina,2001,5652.0,8.639942,0.761125,0.238875,historical
3,ARG,Argentina,2002,6247.0,8.740017,0.759830,0.240170,historical
4,ARG,Argentina,2003,6065.0,8.710455,0.757173,0.242827,historical
5,ARG,Argentina,2004,9796.0,9.189831,0.752349,0.247651,historical
6,ARG,Argentina,2005,6830.0,8.829226,0.750718,0.249282,historical
7,ARG,Argentina,2006,5070.0,8.531293,0.743890,0.256110,historical
8,ARG,Argentina,2007,6994.0,8.852951,0.738162,0.261838,historical
9,ARG,Argentina,2008,7447.0,8.915701,0.737284,0.262716,historical


---
# Part D — Validation & Summary Statistics
## 9 · Descriptive Statistics (Cross-Sectional)

In [73]:
desc_cols = ["uk_import_qty_t", "log_uk_import_qty_t"] + INDICATOR_COLS
cs[desc_cols].describe().round(4)

,uk_import_qty_t,log_uk_import_qty_t,bii,bii_loss
count,37.0000,37.0000,37.0000,37.0000
mean,2863.5603,3.7898,0.7042,0.2958
std,10158.4980,2.8995,0.1457,0.1457
min,0.0200,0.0198,0.3454,0.0000
25%,3.0000,1.2583,0.6129,0.2015
50%,40.8667,2.9636,0.6871,0.3129
75%,642.9643,5.9891,0.7985,0.3871
max,46461.5107,10.7189,1.0000,0.6546


### 9.1 · Correlations with log UK import quantity
We use the **log-transformed** outcome because raw trade volumes are heavily right-skewed.

In [74]:
corr = (
    cs[desc_cols]
    .corr()["log_uk_import_qty_t"]
    .drop(["uk_import_qty_t", "log_uk_import_qty_t"])
    .sort_values(key=abs, ascending=False)
)
print("Pearson correlations with log_uk_import_qty_t:")
print(corr.round(4).to_string())

Pearson correlations with log_uk_import_qty_t:
bii_loss    0.3514
bii        -0.3514


## 10 · Quick OLS Sanity Check (Cross-Sectional)

A preliminary OLS to verify data is usable before moving to full modelling.

- **Model**: `log_import ~ bii_loss`

This tests whether countries with more biodiversity loss export more beef to the UK.
We use **HC3 robust standard errors** to guard against heteroskedasticity from
dominant exporters (Ireland, Brazil).

In [75]:
try:
    import statsmodels.api as sm

    y = cs["log_uk_import_qty_t"].astype(float)

    # -- OLS: bii_loss → log UK beef imports -----------------------------------
    print("OLS: log_uk_import_qty_t ~ bii_loss")
    X = sm.add_constant(cs[["bii_loss"]].astype(float))

    # Standard OLS
    m = sm.OLS(y, X).fit()
    print("\n--- Standard errors ---")
    print(m.summary2().tables[1].round(4))
    print(f"R-sq = {m.rsquared:.4f}  |  Adj-R-sq = {m.rsquared_adj:.4f}  |  N = {int(m.nobs)}")

    # Robust (HC3) standard errors
    m_robust = sm.OLS(y, X).fit(cov_type="HC3")
    print("\n--- Robust (HC3) standard errors ---")
    print(m_robust.summary2().tables[1].round(4))
    print(f"R-sq = {m_robust.rsquared:.4f}  |  Adj-R-sq = {m_robust.rsquared_adj:.4f}  |  N = {int(m_robust.nobs)}")

except ImportError:
    print("statsmodels not installed -- run:  pip install statsmodels")

OLS: log_uk_import_qty_t ~ bii_loss

--- Standard errors ---
           Coef.  Std.Err.       t   P>|t|  [0.025   0.975]
const     1.7219    1.0355  1.6629  0.1053 -0.3802   3.8241
bii_loss  6.9917    3.1489  2.2204  0.0330  0.5991  13.3844
R-sq = 0.1235  |  Adj-R-sq = 0.0984  |  N = 37

--- Robust (HC3) standard errors ---
           Coef.  Std.Err.       z   P>|z|  [0.025   0.975]
const     1.7219    0.9991  1.7234  0.0848 -0.2363   3.6802
bii_loss  6.9917    3.1372  2.2286  0.0258  0.8429  13.1406
R-sq = 0.1235  |  Adj-R-sq = 0.0984  |  N = 37


---
## Summary

**Outcome variable (Y):** `uk_import_qty_t` / `log_uk_import_qty_t`

**Predictor (X):** `bii_loss` (= 1 − BII; higher = more biodiversity loss)

---

### Next Steps for Modelling

1. **Use the extended panel (2001–2024)** with fixed/random effects for more statistical power.
2. **Apply robust standard errors** to guard against heteroskedasticity from dominant exporters.
3. **Run influence diagnostics** (Cook's distance) to check whether results are driven by Ireland or Brazil.
